In [1]:
import pandas as pd
import torch
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
import numpy as np
from sklearn.metrics import mean_absolute_error

In [2]:
data = pd.read_csv('../data/preprocessed.csv', sep='\t', on_bad_lines='warn')
data

,source_id,url_link,city,parsed_at,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,...,has_any_lift,height_m,house_type,renovation,window_type,year_buld,floor,sanuzel_multiple,sanuzel_per_room,description_length
0,7257940100,https://www.avito.ru/samara/kvartiry/1-k._kvar...,Самара,2025-03-24 07:57:37,3500000,101744.190,1,9.0,16.0,5.0,...,1.0,2.0,2,1,1.0,2019.0,3.0,1,1.0,97
1,7253102741,https://www.avito.ru/samara/kvartiry/3-k._kvar...,Самара,2025-03-24 07:57:22,10500000,120689.660,3,14.0,56.0,13.0,...,1.0,2.0,2,2,3.0,2011.0,1.0,1,3.0,65
2,315305891,https://cian.ru/sale/flat/315305891/,Казань,2025-03-24 07:57:13,9000000,195227.766,2,8.2,27.5,10.0,...,1.0,2.8,2,1,1.0,2015.0,9.0,1,2.0,70
3,7233949995,https://www.avito.ru/samara/kvartiry/2-k._kvar...,Самара,2025-03-24 07:57:10,22500000,316011.240,2,32.2,16.7,12.0,...,0.0,2.0,2,3,1.0,2012.0,1.0,1,2.0,105
4,7245785246,https://www.avito.ru/samara/kvartiry/1-k._kvar...,Самара,2025-03-24 07:56:54,1400000,39886.040,1,18.0,11.3,9.0,...,1.0,2.0,2,1,2.0,1978.0,7.0,1,1.0,255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23419,3661338720,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:37:17,10150000,169166.670,2,20.4,54.7,15.0,...,1.0,2.7,0,0,2.0,2015.0,13.0,1,2.0,186
23420,3889002712,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:10:30,5600000,168674.700,1,9.0,18.0,9.0,...,1.0,2.6,4,1,1.0,1958.0,3.0,1,1.0,72
23421,2790563655,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:08:42,7100000,112698.410,3,9.0,39.0,9.0,...,1.0,2.7,4,1,2.0,1984.0,5.0,1,3.0,66
23422,1103958872,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:07:52,4450000,153448.280,1,5.7,38.0,5.0,...,1.0,2.5,2,1,2.0,1967.0,3.0,1,1.0,148


In [3]:
data = data[data.city == 'Нижний Новгород']

In [4]:
data

,source_id,url_link,city,parsed_at,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,...,has_any_lift,height_m,house_type,renovation,window_type,year_buld,floor,sanuzel_multiple,sanuzel_per_room,description_length
13,315101426,https://cian.ru/sale/flat/315101426/,Нижний Новгород,2025-03-24 07:53:20,6390000,199687.500,0,6.0,30.0,19.0,...,1.0,2.8,0,1,1.0,2019.0,17.0,1,0.0,32
14,315102782,https://cian.ru/sale/flat/315102782/,Нижний Новгород,2025-03-24 07:53:03,8700000,153982.301,3,9.0,38.4,9.0,...,1.0,2.8,4,1,2.0,1984.0,5.0,1,3.0,51
15,315104787,https://cian.ru/sale/flat/315104787/,Нижний Новгород,2025-03-24 07:52:40,6500000,185714.286,1,8.5,17.8,5.0,...,0.0,3.0,4,2,2.0,1999.0,4.0,1,1.0,75
16,315105169,https://cian.ru/sale/flat/315105169/,Нижний Новгород,2025-03-24 07:52:24,13100000,132457.027,3,11.5,48.5,10.0,...,1.0,3.0,2,0,2.0,2022.0,2.0,1,3.0,186
17,315162763,https://cian.ru/sale/flat/315162763/,Нижний Новгород,2025-03-24 07:51:10,6800000,149450.549,2,7.0,27.5,9.0,...,1.0,2.5,4,2,2.0,1976.0,7.0,1,2.0,354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23419,3661338720,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:37:17,10150000,169166.670,2,20.4,54.7,15.0,...,1.0,2.7,0,0,2.0,2015.0,13.0,1,2.0,186
23420,3889002712,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:10:30,5600000,168674.700,1,9.0,18.0,9.0,...,1.0,2.6,4,1,1.0,1958.0,3.0,1,1.0,72
23421,2790563655,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:08:42,7100000,112698.410,3,9.0,39.0,9.0,...,1.0,2.7,4,1,2.0,1984.0,5.0,1,3.0,66
23422,1103958872,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:07:52,4450000,153448.280,1,5.7,38.0,5.0,...,1.0,2.5,2,1,2.0,1967.0,3.0,1,1.0,148


In [6]:
features = ['price_object',
        'normalized_price',
        'rooms_count',
        'area_kitchen',
        'area_live',
        'floors_count',
        'distance_to_center',
        'area_live_ratio',
        'area_kitchen_ratio']
X = data[features]

In [7]:
X

,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,distance_to_center,area_live_ratio,area_kitchen_ratio
13,6390000,199687.500,0,6.0,30.0,19.0,7.333176,0.937500,0.187500
14,8700000,153982.301,3,9.0,38.4,9.0,6.317608,0.679646,0.159292
15,6500000,185714.286,1,8.5,17.8,5.0,9.961762,0.508571,0.242857
16,13100000,132457.027,3,11.5,48.5,10.0,5.171610,0.490394,0.116279
17,6800000,149450.549,2,7.0,27.5,9.0,10.386619,0.604396,0.153846
...,...,...,...,...,...,...,...,...,...
23419,10150000,169166.670,2,20.4,54.7,15.0,4.751295,0.911667,0.340000
23420,5600000,168674.700,1,9.0,18.0,9.0,1.862286,0.542169,0.271084
23421,7100000,112698.410,3,9.0,39.0,9.0,12.740409,0.619048,0.142857
23422,4450000,153448.280,1,5.7,38.0,5.0,3.015731,1.310345,0.196552


In [8]:
y = data['price_object']

In [9]:
model = GradientBoostingRegressor(loss='quantile', alpha=0.5, random_state=42)

In [10]:
model.fit(X, y)
pred = model.predict(X)

In [11]:
def smape(y_true, y_pred):
    return 100 * np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [12]:
smp = smape(y, pred)
print("SMAPE:", smp)

SMAPE: 0.8043631748035098


In [13]:
y

13        6390000
14        8700000
15        6500000
16       13100000
17        6800000
           ...   
23419    10150000
23420     5600000
23421     7100000
23422     4450000
23423     3950000
Name: price_object, Length: 18728, dtype: int64

In [14]:
pred

array([6479019.03379557, 8517420.60246039, 6479019.03379557, ...,
       7201174.59100539, 4417935.89379767, 3934110.73085441])

In [15]:
isf = IsolationForest(contamination=0.1)

data_pred = data.copy()
isf.fit(X)

predictions = isf.score_samples(X)

data_pred['isf_0.1'] = predictions

In [16]:
data_pred['quantile'] = y < pred

In [17]:
data_pred

,source_id,url_link,city,parsed_at,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,...,house_type,renovation,window_type,year_buld,floor,sanuzel_multiple,sanuzel_per_room,description_length,isf_0.1,quantile
13,315101426,https://cian.ru/sale/flat/315101426/,Нижний Новгород,2025-03-24 07:53:20,6390000,199687.500,0,6.0,30.0,19.0,...,0,1,1.0,2019.0,17.0,1,0.0,32,-0.484471,True
14,315102782,https://cian.ru/sale/flat/315102782/,Нижний Новгород,2025-03-24 07:53:03,8700000,153982.301,3,9.0,38.4,9.0,...,4,1,2.0,1984.0,5.0,1,3.0,51,-0.386743,False
15,315104787,https://cian.ru/sale/flat/315104787/,Нижний Новгород,2025-03-24 07:52:40,6500000,185714.286,1,8.5,17.8,5.0,...,4,2,2.0,1999.0,4.0,1,1.0,75,-0.394939,False
16,315105169,https://cian.ru/sale/flat/315105169/,Нижний Новгород,2025-03-24 07:52:24,13100000,132457.027,3,11.5,48.5,10.0,...,2,0,2.0,2022.0,2.0,1,3.0,186,-0.412289,True
17,315162763,https://cian.ru/sale/flat/315162763/,Нижний Новгород,2025-03-24 07:51:10,6800000,149450.549,2,7.0,27.5,9.0,...,4,2,2.0,1976.0,7.0,1,2.0,354,-0.357407,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23419,3661338720,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:37:17,10150000,169166.670,2,20.4,54.7,15.0,...,0,0,2.0,2015.0,13.0,1,2.0,186,-0.507807,False
23420,3889002712,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:10:30,5600000,168674.700,1,9.0,18.0,9.0,...,4,1,1.0,1958.0,3.0,1,1.0,72,-0.394558,True
23421,2790563655,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:08:42,7100000,112698.410,3,9.0,39.0,9.0,...,4,1,2.0,1984.0,5.0,1,3.0,66,-0.391218,True
23422,1103958872,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:07:52,4450000,153448.280,1,5.7,38.0,5.0,...,2,1,2.0,1967.0,3.0,1,1.0,148,-0.477846,False


In [18]:
data_pred.loc[data_pred['quantile'] == False, 'isf_0.1'] = -1 * data_pred.loc[data_pred['quantile'] == False, 'isf_0.1']

In [19]:
data_pred

,source_id,url_link,city,parsed_at,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,...,house_type,renovation,window_type,year_buld,floor,sanuzel_multiple,sanuzel_per_room,description_length,isf_0.1,quantile
13,315101426,https://cian.ru/sale/flat/315101426/,Нижний Новгород,2025-03-24 07:53:20,6390000,199687.500,0,6.0,30.0,19.0,...,0,1,1.0,2019.0,17.0,1,0.0,32,-0.484471,True
14,315102782,https://cian.ru/sale/flat/315102782/,Нижний Новгород,2025-03-24 07:53:03,8700000,153982.301,3,9.0,38.4,9.0,...,4,1,2.0,1984.0,5.0,1,3.0,51,0.386743,False
15,315104787,https://cian.ru/sale/flat/315104787/,Нижний Новгород,2025-03-24 07:52:40,6500000,185714.286,1,8.5,17.8,5.0,...,4,2,2.0,1999.0,4.0,1,1.0,75,0.394939,False
16,315105169,https://cian.ru/sale/flat/315105169/,Нижний Новгород,2025-03-24 07:52:24,13100000,132457.027,3,11.5,48.5,10.0,...,2,0,2.0,2022.0,2.0,1,3.0,186,-0.412289,True
17,315162763,https://cian.ru/sale/flat/315162763/,Нижний Новгород,2025-03-24 07:51:10,6800000,149450.549,2,7.0,27.5,9.0,...,4,2,2.0,1976.0,7.0,1,2.0,354,0.357407,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23419,3661338720,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:37:17,10150000,169166.670,2,20.4,54.7,15.0,...,0,0,2.0,2015.0,13.0,1,2.0,186,0.507807,False
23420,3889002712,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:10:30,5600000,168674.700,1,9.0,18.0,9.0,...,4,1,1.0,1958.0,3.0,1,1.0,72,-0.394558,True
23421,2790563655,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:08:42,7100000,112698.410,3,9.0,39.0,9.0,...,4,1,2.0,1984.0,5.0,1,3.0,66,-0.391218,True
23422,1103958872,https://www.avito.ru/nizhniy_novgorod/kvartiry...,Нижний Новгород,2024-04-20 03:07:52,4450000,153448.280,1,5.7,38.0,5.0,...,2,1,2.0,1967.0,3.0,1,1.0,148,0.477846,False


In [20]:
data_pred[data_pred['isf_0.1'] == data_pred['isf_0.1'].min()].values

array([[3201477771,
        'https://www.avito.ru/nizhniy_novgorod/kvartiry/kvartira-studiya_209m_414et._3201477771',
        'Нижний Новгород', '2024-06-25 03:19:24', 5900000, 282296.65, 0,
        20.0, 58.0, 14.0, 3.0694236817129883, 2.7751196172248807,
        0.9569377990430624, 20.9, 1.0, 1.0, 2.7, 0, 1, 2.0, 2019.0, 4.0,
        1, 0.0, 73, -0.698688704591284, True]], dtype=object)

In [21]:
data_pred.sort_values('isf_0.1')

,source_id,url_link,city,parsed_at,price_object,normalized_price,rooms_count,area_kitchen,area_live,floors_count,...,house_type,renovation,window_type,year_buld,floor,sanuzel_multiple,sanuzel_per_room,description_length,isf_0.1,quantile
15373,3201477771,https://www.avito.ru/nizhniy_novgorod/kvartiry/kvartira-studiya_209m_414et._3201477771,Нижний Новгород,2024-06-25 03:19:24,5900000,282296.650,0,20.0,58.0,14.0,...,0,1,2.0,2019.0,4.0,1,0.0,73,-0.698689,True
7616,306634177,https://nn.cian.ru/sale/flat/306634177/,Нижний Новгород,2024-08-31 18:46:33,37000000,231250.000,5,40.0,90.0,2.0,...,1,3,1.0,1917.0,2.0,1,5.0,104,-0.685749,True
15880,303633092,https://nn.cian.ru/sale/flat/303633092/,Нижний Новгород,2024-06-20 14:30:29,35000000,140000.000,5,45.0,200.0,17.0,...,2,0,3.0,1995.0,16.0,1,5.0,185,-0.684399,True
19081,302150685,https://nn.cian.ru/sale/flat/302150685/,Нижний Новгород,2024-05-18 19:08:20,3491250,205367.647,0,18.0,51.0,10.0,...,4,2,2.0,2015.0,1.0,1,0.0,181,-0.682319,True
8162,306365475,https://nn.cian.ru/sale/flat/306365475/,Нижний Новгород,2024-08-23 13:17:40,3600000,186528.497,0,16.0,47.0,19.0,...,4,1,1.0,2018.0,17.0,1,0.0,146,-0.678252,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12990,304415758,https://nn.cian.ru/sale/flat/304415758/,Нижний Новгород,2024-07-09 09:23:52,65000000,302325.581,4,70.4,83.0,19.0,...,3,0,3.0,2018.0,19.0,1,4.0,226,0.710659,False
23061,2210283316,https://www.avito.ru/nizhniy_novgorod/kvartiry/4-k._kvartira_225m_2222et._2210283316,Нижний Новгород,2024-04-22 13:33:07,81862500,363833.330,4,50.0,160.0,22.0,...,3,1,2.0,2006.0,22.0,1,4.0,295,0.716771,False
23063,2210164161,https://www.avito.ru/nizhniy_novgorod/kvartiry/4-k._kvartira_1979m_2222et._2210164161,Нижний Новгород,2024-04-22 13:32:36,74662500,377273.880,4,50.0,120.0,22.0,...,3,1,2.0,2006.0,22.0,1,4.0,295,0.718228,False
14803,4134223390,https://www.avito.ru/nizhniy_novgorod/kvartiry/5-k._kvartira_240m_56et._4134223390,Нижний Новгород,2024-06-29 12:38:53,120000000,500000.000,5,25.0,173.0,6.0,...,2,3,3.0,2007.0,5.0,1,5.0,273,0.722873,False
